# Voxel modulation

Port of `packages/niivue/examples/vox.modulate.html`. Loads FA and V1 volumes, then switches between scalar, vector, and FA-modulated V1 display modes.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

from ipyniivue import NiiVue

BASE_URL = "https://niivue.github.io/mono"
VOLUMES = f"{BASE_URL}/volumes"
MESHES = f"{BASE_URL}/meshes"

nv = NiiVue(
    background_color=[0.0, 0.0, 0.2, 1.0],
    is_3d_crosshair_visible=True,
    crosshair_width=0.5,
    is_snap_to_voxel_centers=True,
    volume_is_nearest_interpolation=True,
    backend="webgl2",
)

mode = widgets.Dropdown(
    options=[
        ("FA", 0),
        ("V1", 1),
        ("V1 modulated by FA", 2),
        ("V1 slice shader", 3),
        ("V1 modulated by FA, slice shader", 4),
    ],
    value=4,
    description="Display",
)
fa_range = widgets.FloatRangeSlider(
    value=[0.0, 1.0], min=0.0, max=1.0, step=0.01, description="FA range"
)
clip_dark = widgets.Checkbox(value=True, description="Clip dark")


def update_fa_range(change):
    mn, mx = change["new"]
    nv.set_volume(0, {"calMin": min(mn, mx), "calMax": max(mn, mx)})


def update_clip_dark(change):
    nv.volume_is_alpha_clip_dark = change["new"]


def update_mode(change=None):
    idx = mode.value
    if idx == 0:
        nv.set_volume(0, {"opacity": 1.0})
        nv.set_volume(1, {"opacity": 0.0})
    elif idx <= 2:
        nv.set_volume(0, {"opacity": 0.0})
        nv.set_volume(1, {"opacity": 1.0})
    else:
        nv.set_volume(0, {"opacity": 1.0})
        nv.set_volume(1, {"opacity": 1.0})

    if idx in (2, 4):
        nv.set_modulation_image("V1", "FA")
    else:
        nv.set_modulation_image("V1", "")
    nv.volume_is_v1_slice_shader = idx > 2


fa_range.observe(update_fa_range, names="value")
clip_dark.observe(update_clip_dark, names="value")
mode.observe(update_mode, names="value")

controls = widgets.VBox([widgets.HBox([mode, clip_dark]), fa_range])
display(controls)
display(nv)

nv.load_volumes(
    [
        {"url": f"{VOLUMES}/FA.nii.gz", "name": "FA", "id": "FA", "opacity": 1},
        {"url": f"{VOLUMES}/V1.nii.gz", "name": "V1", "id": "V1", "opacity": 1},
    ]
)
update_fa_range({"new": fa_range.value})
update_clip_dark({"new": clip_dark.value})
update_mode()